# Создание моделей для предсказания свойств углепластика, полученного по вакуумной технологии

In [1]:
import pandas as pd
import numpy as np


# инструменты для построения модели:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # инструмент для создания и обучения модели
from sklearn.ensemble import RandomForestRegressor # инструмент для создания и обучения модели
from sklearn import metrics # инструменты для оценки точности модели
from xgboost import XGBRegressor

import pickle

RANDOM_SEED = 42

In [2]:
df = pd.read_csv('data/dataset_prepaired.csv')
df.head()

,Linera density,Density yarn,Strength Gpa,Module Gpa,lengthening,Mass size,breaking the loop,Surface density of the fabric,Prepreg surface density,Resin content,viscosity,Gelation time,Resin Tg,technology,Thickness of the monolayer,density,Strength_plastik,Module_plastik,LSS,Plastik_Tg
0,188.0,1.758,4.59,253.0,1.814229,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
1,189.0,1.758,4.48,260.0,1.723077,1.1,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
2,188.0,1.759,4.28,257.0,1.665370,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
3,187.0,1.758,4.77,256.0,1.863281,1.0,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0
4,190.0,1.757,4.56,255.0,1.788235,0.9,24.3,196.0,319.4,38.63,23.672,19.0,150.2,0,0.209,1.536,879.0,70.1,77.2,164.0


In [3]:
df = df[df['technology'] == 0]
df = df.drop('technology', axis=1)
#df_autoclave = df[df['technology'] == 1]

In [4]:
df = df.rename(columns={'Thickness of the monolayer' : 'Thickness_monolayer',
                        'Module_plastik ' : 'Module_plastik'})

In [5]:
X = np.array([[190, 1.779, 4.6, 265, 1.7, 1.3, 20, 210, 333, 37.40, 32.33, 12.8, 149.0]])

### Модель для предсказания толщины монослоя Thickness_monolayer

In [6]:
train_data_thickness = df.drop(['density', 'Strength_plastik', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_thickness.drop(['Thickness_monolayer'], axis=1))
y = np.array(train_data_thickness.Thickness_monolayer.values)

In [7]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

In [8]:
# НАСТРОЙКИ 
model_rf_thikness = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

In [9]:
# обучаем модель на тестовом наборе данных
model_rf_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_thikness.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [10]:
def mean_absolute_percentage_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr) / y_tr)) * 100

In [11]:
# сравниваем предсказанные значения (y_pred) с реальными (y_test), 
# метрика mean squared error, MSE показывает среднеквадратичное отклонение:

def mean_squared_error(y_tr, y_pr):
    """Получение средней абсолютной ошибки"""
    y_tr, y_pr = np.array(y_tr), np.array(y_pr)
    return np.mean(np.abs((y_tr - y_pr)**2)) 

In [12]:
print('model_rf_thikness MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_thikness MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_thikness MSE: 0.0
model_rf_thikness MAPE: 0.003


In [13]:
# save model
with open('vac_model_rf_thikness.pkl','wb') as f:
    pickle.dump(model_rf_thikness,f)

In [14]:
model_lr_thikness = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_thikness.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_thikness.predict(X_test)

print('model_lr_thikness MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_thikness MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_thikness MSE: 0.0
model_lr_thikness MAPE: 0.607


In [15]:
# save
with open('vac_model_lr_thikness.pkl','wb') as f:
    pickle.dump(model_lr_thikness,f)

In [16]:
print('Линейная регрессия. Толщина монослоя:', model_lr_thikness.predict(X))
print('Случайный лес регрессия. Толщина монослоя:', model_rf_thikness.predict(X))

Линейная регрессия. Толщина монослоя: [0.20968481 0.21141898 0.20968576 ... 0.22387731 0.22438837 0.2238029 ]
Случайный лес регрессия. Толщина монослоя: [0.209  0.209  0.209  ... 0.2235 0.2235 0.2235]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


### Модель для предсказания плотности углепластика density

In [17]:
train_data_density = df.drop(['Thickness_monolayer', 'Strength_plastik', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_density.drop(['density'], axis=1))
y = np.array(train_data_density.density.values)

In [18]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_density = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_density.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [19]:
print('model_rf_density MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_densityMAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_density MSE: 0.0
model_rf_densityMAPE: 0.002


In [20]:
# save
with open('vac_model_rf_density.pkl','wb') as f:
    pickle.dump(model_rf_density,f)

In [21]:
model_lr_density = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_density.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_density.predict(X_test)

print('model_lr_density MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_density MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_density MSE: 0.0
model_lr_density MAPE: 0.236


In [22]:
# save
with open('vac_model_lr_density.pkl','wb') as f:
    pickle.dump(model_lr_density,f)

In [23]:
print('Линейная регрессия. Плотность:', model_lr_density.predict(X))
print('Случайный лес регрессия. Плотность:', model_rf_density.predict(X))

Линейная регрессия. Плотность: [1.52607393 1.52738504 1.52607237 ... 1.52191354 1.52039419 1.52099974]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.


Случайный лес регрессия. Плотность: [1.536  1.536  1.536  ... 1.5215 1.5215 1.5215]


[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


### Модель для предсказания прочности углепластика Strength

In [54]:
train_data_strength = df.drop(['Thickness_monolayer', 'density', 'Module_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_strength.drop(['Strength_plastik'], axis=1))
y = np.array(train_data_strength.Strength_plastik.values)

In [59]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_strength = RandomForestRegressor(
    n_estimators=500, 
    max_features=4,
    min_samples_leaf=6,
    max_depth=8,
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_strength.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:    0.3s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.1s finished


In [60]:
print('model_rf_strength MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_strength MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_strength MSE: 4.854
model_rf_strength MAPE: 0.023


In [27]:
# save
with open('vac_model_rf_strength.pkl','wb') as f:
    pickle.dump(model_rf_strength,f)

In [28]:
model_lr_strength = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_strength.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_strength.predict(X_test)

print('model_lr_strength MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_strengthMAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_strength MSE: 49.333
model_lr_strengthMAPE: 0.609


In [29]:
# save
with open('vac_model_lr_strength.pkl','wb') as f:
    pickle.dump(model_lr_strength,f)

In [30]:
model_xgb_strenght = XGBRegressor(lerning_rate = 0.01)

# обучаем модель на тестовом наборе данных
model_xgb_strenght.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_xgb_strenght.predict(X_test)

print('model_xgb_strenght MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_xgb_strenght MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_xgb_strenght MSE: 3.63
model_xgb_strenght MAPE: 0.014


/home/alexandr/anaconda3/envs/ptn312/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [00:54:49] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "lerning_rate" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [31]:
# save
with open('vac_model_xgb_strenght.pkl','wb') as f:
    pickle.dump(model_xgb_strenght,f)

In [32]:
print('Линейная регрессия. Прочность:', model_lr_strength.predict(X))
print('Случайный лес регрессия. Прочность:', model_rf_strength.predict(X))
print('XGB регрессия. Прочность:', model_xgb_strenght.predict(X))

Линейная регрессия. Прочность: [873.12636396 873.06376135 873.34849312 ... 897.7394274  896.87413256
 898.23509749]
Случайный лес регрессия. Прочность: [879.00812036 878.99512648 879.00388905 ... 898.99822905 898.97263634
 898.96401127]
XGB регрессия. Прочность: [879.05225 878.90564 878.99524 ... 898.99945 898.99884 898.99817]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 500 out of 500 | elapsed:    0.0s finished


### Модель для предсказания модуля углепластика Module

In [33]:
train_data_module = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik',
       'LSS', 'Plastik_Tg'], axis=1)

X = np.array(train_data_module.drop(['Module_plastik'], axis=1))
y = np.array(train_data_module.Module_plastik .values)

In [34]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_module = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_module.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [35]:
print('model_rf_module MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_module MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_module MSE: 0.002
model_rf_module MAPE: 0.003


In [36]:
# save
with open('vac_model_rf_module.pkl','wb') as f:
    pickle.dump(model_rf_module,f)

In [37]:
model_lr_module = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_module.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_module.predict(X_test)

print('model_lr_moduleMSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_module MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_moduleMSE: 1.077
model_lr_module MAPE: 1.127


In [38]:
# save
with open('vac_model_lr_module.pkl','wb') as f:
    pickle.dump(model_lr_module,f)

In [39]:
print('Линейная регрессия. Модуль упругости:', model_lr_module.predict(X))
print('Случайный лес регрессия. Модуль упругости:', model_rf_module.predict(X))

Линейная регрессия. Модуль упругости: [70.72563387 69.51150458 70.69819137 ... 70.84163081 70.65346221
 70.88405085]
Случайный лес регрессия. Модуль упругости: [70.1 70.1 70.1 ... 71.  71.  71. ]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


### Модель для предсказания межслоевой прочности углепластика LSS

In [40]:
train_data_lss = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik',
                             'Plastik_Tg'], axis=1)

X = np.array(train_data_lss.drop(['LSS'], axis=1))
y = np.array(train_data_lss.LSS.values)

In [41]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_lss = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_lss.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [42]:
print('model_rf_lss MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_lss MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_lss MSE: 0.025
model_rf_lss MAPE: 0.01


In [43]:
# save
with open('vac_model_rf_lss.pkl','wb') as f:
    pickle.dump(model_rf_lss,f)

In [44]:
model_lr_lss = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_lss.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_lss.predict(X_test)

print('model_lr_lss MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_lssMAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_lss MSE: 1.779
model_lr_lssMAPE: 1.388


In [45]:
# save
with open('vac_model_lr_lss.pkl','wb') as f:
    pickle.dump(model_lr_lss,f)

In [46]:
print('Линейная регрессия. Прочность при сдвиге:', model_lr_lss.predict(X))
print('Случайный лес регрессия. Прочность при сдвиге:', model_rf_lss.predict(X))

Линейная регрессия. Прочность при сдвиге: [74.53659588 75.56339713 74.48346683 ... 86.92895948 86.6285063
 86.44990935]
Случайный лес регрессия. Прочность при сдвиге: [77.2  77.2  77.2  ... 86.45 86.45 86.45]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


# Модель для предсказания температуры стеклования углепластика Tg

In [47]:
train_data_Tg = df.drop(['Thickness_monolayer', 'density', 'Strength_plastik', 'Module_plastik', 
                         'LSS'], axis=1)

X = np.array(train_data_Tg.drop(['Plastik_Tg'], axis=1))
y = np.array(train_data_Tg.Plastik_Tg.values)

In [48]:
# воспользуемся специальной функцие train_test_split для разделения тестовых данных
# выделим 20% данных на валидацию (параметр test_size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED)

# НАСТРОЙКИ 
model_rf_Tg = RandomForestRegressor(
    n_estimators=100, 
    verbose=1, 
    n_jobs=-1, 
    random_state=RANDOM_SEED)

# обучаем модель на тестовом наборе данных
model_rf_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_rf_Tg.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


In [49]:
print('model_rf_TgMSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_rf_Tg MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_rf_TgMSE: 0.021
model_rf_Tg MAPE: 0.006


In [50]:
# save
with open('vac_model_rf_Tg.pkl','wb') as f:
    pickle.dump(model_rf_Tg,f)

In [51]:
model_lr_Tg = LinearRegression()

# обучаем модель на тестовом наборе данных
model_lr_Tg.fit(X_train, y_train)

# предсказанные значения записываем в переменную y_pred
y_pred = model_lr_Tg.predict(X_test)

print('model_lr_Tg MSE:', round(mean_squared_error(y_test, y_pred), 3))

print('model_lr_Tg MAPE:', round(mean_absolute_percentage_error(y_test, y_pred), 3))

model_lr_Tg MSE: 0.252
model_lr_Tg MAPE: 0.213


In [52]:
# save
with open('vac_model_lr_Tg.pkl','wb') as f:
    pickle.dump(model_lr_Tg,f)

In [53]:
print('Линейная регрессия. Температура стеклования:', model_lr_Tg.predict(X))
print('Случайный лес регрессия. Температура стеклования:', model_rf_Tg.predict(X))

Линейная регрессия. Температура стеклования: [163.30543953 164.03471098 163.3081314  ... 163.15631938 163.19797627
 163.06549736]


[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


Случайный лес регрессия. Температура стеклования: [164. 164. 164. ... 163. 163. 163.]
